# 01 — Data Profiling

Profiling **ID_REG_Parsed — Indonesian Regulation Parsed Dataset** dengan Polars Lazy API. Satu baris merepresentasikan pasal, ayat, klausul, atau bagian regulasi Indonesia.

In [1]:
import polars as pl
from pathlib import Path
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)
print('Polars version:', pl.__version__)

Polars version: 1.44.2


## 1. Lokasi dan ukuran dataset

In [2]:
CWD = Path.cwd().resolve()
if (CWD / 'data').exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / 'data').exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError('Root project tidak ditemukan.')

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'DataSet'
PARQUET_FILES = sorted(RAW_DIR.glob('*.parquet'))
if not PARQUET_FILES:
    raise FileNotFoundError(f'Tidak ada file Parquet di {RAW_DIR}')
RAW_GLOB = str(RAW_DIR / '*.parquet')
total_size_bytes = sum(p.stat().st_size for p in PARQUET_FILES)
total_size_mb = total_size_bytes / 1024**2
total_size_gb = total_size_bytes / 1024**3
print('RAW_DIR:', RAW_DIR)
print('Jumlah file:', len(PARQUET_FILES))
print(f'Ukuran: {total_size_gb:.2f} GB ({total_size_mb:,.2f} MB)')

RAW_DIR: D:\bigdata-kuliah\tugas1-FakhriZiddanA\data\raw\DataSet
Jumlah file: 64
Ukuran: 1.47 GB (1,507.97 MB)


## 2. Lazy scan, schema, dan contoh data

In [3]:
lf = pl.scan_parquet(RAW_GLOB)
schema = lf.collect_schema()
columns = schema.names()
print(f'Jumlah kolom: {len(columns)}')
for name, dtype in schema.items():
    print(f'- {name}: {dtype}')
lf.head(10).collect()

Jumlah kolom: 7
- Regulation Name: String
- Regulation Number: String
- Year: String
- About: String
- Chapter: String
- Article: String
- Content: String


Regulation Name,Regulation Number,Year,About,Chapter,Article,Content
str,str,str,str,str,str,str
"""PERATURAN PEMERINTAH REPUBL IK…","""13""","""2009""","""JENIS DAN TARIF ATAS JENIS PEN…",null,"""Pasal 1""","""(1)Jenis Penerimaan Negara Buk…"
"""PERATURAN PEMERINTAH REPUBL IK…","""13""","""2009""","""JENIS DAN TARIF ATAS JENIS PEN…",null,"""Pasal 2""","""Jenis Penerimaan Negara Bukan …"
"""PERATURAN PEMERINTAH REPUBL IK…","""13""","""2009""","""JENIS DAN TARIF ATAS JENIS PEN…",null,"""Pasal 3""","""... PRESIDEN REPUBLIK INDONESI…"
"""PERATURAN PEMERINTAH REPUBL IK…","""13""","""2009""","""JENIS DAN TARIF ATAS JENIS PEN…",null,"""Pasal 3""","""(1)Tarif atas jenis Penerimaan…"
"""PERATURAN PEMERINTAH REPUBL IK…","""13""","""2009""","""JENIS DAN TARIF ATAS JENIS PEN…",null,"""Pasal 4""","""(1)Tarif atas jenis Penerimaan…"
"""PERATURAN PEMERINTAH REPUBL IK…","""13""","""2009""","""JENIS DAN TARIF ATAS JENIS PEN…",null,"""Pasal 5""","""... PRESIDEN REPUBLIK INDONESI…"
"""PERATURAN PEMERINTAH REPUBL IK…","""13""","""2009""","""JENIS DAN TARIF ATAS JENIS PEN…",null,"""Pasal 5""","""Seluruh Pen erimaan Negara Buk…"
"""PERATURAN PEMERINTAH REPUBL IK…","""13""","""2009""","""JENIS DAN TARIF ATAS JENIS PEN…",null,"""Pasal 6""","""Pada saat Peraturan Pemerintah…"
"""PERATURAN PEMERINTAH REPUBL IK…","""13""","""2009""","""JENIS DAN TARIF ATAS JENIS PEN…",null,"""Pasal 7""","""Peraturan Pemerintah ini mulai…"


## 3. Skala, missing values, dan nilai unik

In [4]:
total_rows = lf.select(pl.len()).collect().item()
summary = pl.DataFrame({
    'metrik': ['jumlah_file', 'ukuran_mb', 'ukuran_gb', 'jumlah_baris', 'jumlah_kolom'],
    'nilai': [float(len(PARQUET_FILES)), round(total_size_mb, 2), round(total_size_gb, 2), float(total_rows), float(len(columns))],
})
summary

metrik,nilai
str,f64
"""jumlah_file""",64.0
"""ukuran_mb""",1507.97
"""ukuran_gb""",1.47
"""jumlah_baris""",3.630932e6
"""jumlah_kolom""",7.0


In [5]:
# Profiling dari sampel agar notebook tetap ringan pada dataset besar.
PROFILE_SAMPLE_ROWS = 100_000
profile_lf = lf.limit(PROFILE_SAMPLE_ROWS)
null_counts = profile_lf.select(
    pl.len().alias('__sample_rows'),
    *[pl.col(c).null_count().alias(c) for c in columns],
).collect(engine='streaming')
sample_rows = null_counts['__sample_rows'][0]
null_profile = (
    null_counts.drop('__sample_rows')
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns((pl.col('jumlah_null') / sample_rows * 100).round(4).alias('persentase_null'))
    .sort('jumlah_null', descending=True)
)
null_profile

kolom,jumlah_null,persentase_null
str,u32,f64
"""Chapter""",16236,16.236
"""About""",9432,9.432
"""Regulation Name""",399,0.399
"""Regulation Number""",46,0.046
"""Content""",24,0.024
"""Year""",0,0.0
"""Article""",0,0.0


In [6]:
important_columns = [c for c in ['Regulation Name', 'Regulation Number', 'Year', 'About', 'Chapter', 'Article'] if c in columns]
# approx_n_unique jauh lebih hemat memori daripada n_unique() exact.
unique_profile = profile_lf.select([pl.col(c).approx_n_unique().alias(c) for c in important_columns]).collect(engine='streaming')
unique_profile

Regulation Name,Regulation Number,Year,About,Chapter,Article
u32,u32,u32,u32,u32,u32
1070,678,70,3291,7712,1056


## 4. Potensi duplikasi

> Pemeriksaan ini memakai sampel metadata (tanpa kolom teks `Content`) agar aman untuk laptop dengan memori terbatas. Baris belum dihapus; penanganannya dilakukan dalam notebook cleaning.

In [7]:
# Jangan hitung n_unique() seluruh file/kolom Content: sangat berat untuk teks besar.
DUPLICATE_SAMPLE_ROWS = 50_000
identity_columns = [c for c in ['Regulation Name', 'Regulation Number', 'Year', 'Chapter', 'Article'] if c in columns]
duplicate_lf = lf.select(identity_columns).limit(DUPLICATE_SAMPLE_ROWS)
duplicate_profile = duplicate_lf.select([
    pl.len().alias('baris_sampel'),
    (pl.len() - pl.struct(identity_columns).n_unique()).alias('potensi_duplikat_metadata'),
]).collect(engine='streaming')
duplicate_profile

baris_sampel,potensi_duplikat_metadata
u32,u32
50000,4490


## 5. Validasi dan distribusi tahun regulasi

In [8]:
if 'Year' not in columns:
    raise KeyError("Kolom 'Year' tidak ditemukan.")
lf_profile = lf.with_columns(pl.col('Year').cast(pl.String).str.strip_chars().cast(pl.Int32, strict=False).alias('tahun'))
year_profile = lf_profile.select([
    pl.col('tahun').min().alias('tahun_awal'),
    pl.col('tahun').max().alias('tahun_akhir'),
    pl.col('tahun').null_count().alias('tahun_tidak_valid'),
    pl.col('tahun').filter((pl.col('tahun') < 1700) | (pl.col('tahun') > 2100)).count().alias('tahun_di_luar_rentang_wajar'),
]).collect()
year_profile

tahun_awal,tahun_akhir,tahun_tidak_valid,tahun_di_luar_rentang_wajar
i32,i32,u32,u32
1945,2030,580,0


In [9]:
units_per_year = (lf_profile.filter(pl.col('tahun').is_not_null()).group_by('tahun').agg([pl.len().alias('jumlah_unit'), pl.col('Regulation Number').n_unique().alias('jumlah_nomor_regulasi')]).sort('tahun').collect())
units_per_year

tahun,jumlah_unit,jumlah_nomor_regulasi
i32,u32,u32
1945,206666,1064
1946,332,24
1947,699,45
1948,1255,79
1949,371,32
1950,30882,449
1951,1877,83
1952,852,53
1953,1619,81


## 6. Panjang teks dan distribusi regulasi

In [10]:
length_exprs = []
for column, prefix in [('About', 'about'), ('Content', 'content')]:
    if column in columns:
        length = pl.col(column).cast(pl.String).str.len_chars()
        length_exprs += [length.min().alias(f'{prefix}_min'), length.mean().round(2).alias(f'{prefix}_mean'), length.median().alias(f'{prefix}_median'), length.max().alias(f'{prefix}_max')]
lf.select(length_exprs).collect()

about_min,about_mean,about_median,about_max,content_min,content_mean,content_median,content_max
u32,f64,f64,u32,u32,f64,f64,u32
3,43.42,43.0,18301,11,1040.64,428.0,32767


In [11]:
regulation_type_profile = (lf.group_by('Regulation Name').agg(pl.len().alias('jumlah_unit')).sort('jumlah_unit', descending=True).head(20).collect())
regulation_type_profile

Regulation Name,jumlah_unit
str,u32
"""PERATURAN PEMERINTAH REPUBLIK …",85545
"""PERATURAN""",57055
"""UNDANG-UNDANG REPUBLIK INDONES…",33790
"""PERATURAN MENTER! KEUANGAN REP…",30544
"""PERATURAN PRESIDEN REPUBLIK IN…",29327
"""PERATURAN DAERAH""",26586
"""PERATURAN MENTERI KEUANGAN REP…",23991
"""KEPUTUSAN PRESIDEN REPUBLIK IN…",20535
"""UNDANG -UNDANG REPUBLIK INDONE…",19173


## 7. Ringkasan kelayakan dataset

In [12]:
meets_rows = total_rows > 1_000_000
meets_size = total_size_mb >= 500
print('=== RINGKASAN PROFILING ===')
print(f'Jumlah file     : {len(PARQUET_FILES):,}')
print(f'Ukuran dataset  : {total_size_gb:.2f} GB')
print(f'Jumlah baris    : {total_rows:,}')
print(f'Jumlah kolom    : {len(columns)}')
print(f'> 1 juta baris  : {"YA" if meets_rows else "TIDAK"}')
print(f'>= 500 MB       : {"YA" if meets_size else "TIDAK"}')
print(f'Memenuhi syarat : {"YA" if meets_rows or meets_size else "TIDAK"}')

=== RINGKASAN PROFILING ===
Jumlah file     : 64
Ukuran dataset  : 1.47 GB
Jumlah baris    : 3,630,932
Jumlah kolom    : 7
> 1 juta baris  : YA
>= 500 MB       : YA
Memenuhi syarat : YA


## Kesimpulan

Dataset dipindai dengan `scan_parquet()` agar tidak dimuat sekaligus ke memori. Hasil profiling menjadi dasar cleaning: null pada metadata/isi, tahun tidak valid, unit duplikat, teks ekstrem, dan konsistensi penulisan nama, nomor, bab, serta pasal regulasi.